# Silver Layer Incremental Industrial V2
Incremental Silver Layer processing using only new Bronze rows

## Step 1 -- Imports and Setup
This cell imports Spark, Window and Delta helpers, switches to the right catalog, makes sure the **Silver Schema** exists and creates a `silver_run_id` for the current run.

In [0]:
from pyspark.sql import functions as f
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime
import uuid

In [0]:
spark.sql("use catalog ecomdatabricks")
spark.sql("create schema if not exists silver_schema")

silver_run_id = str(uuid.uuid4())

print("Current Silver run id: ", silver_run_id)

## Step 2 -- Silver control table

This table stores the latest Silver processing state for each entity.

It helps us track:
- the latest Bronze run already processed by Silver.
- the latest Bronze ingestion timestamp already processed.
- how many rows were merged in the latest Silver run.

In [0]:
spark.sql("""
            create table if not exists ecomdatabricks.silver_schema.processing_control(
              layer string,
              entity_name string,
              last_processed_bronze_run_id string,
              last_processed_bronze_ingested_at timestamp,
              rows_merged bigint,
              run_status string,
              silver_run_id string,
              updated_at timestamp
            )
            using delta
          
          """)

## Step 3 -- Helper functions
This cell contains reusable logic for silver:

- `upsert_to_silver()` merges cleaned / transformed rows into the Silver target table,
- `get_last_processed_bronze_ingestion_at()` reads the Silver watermark,
- `upsert_silver_control()` updates the Silver Control table,
- `get_incremental_bronze()` reads only new Bronze rows that Silver has not processed yet.

In [0]:
def upsert_to_silver(df_source, target_table, join_key):
  if spark.catalog.tableExists(target_table):
    dt = DeltaTable.forName(spark, target_table)
    (
      dt.alias("target")
      .merge(df_source.alias("source"), f"target.{join_key} = source.{join_key}")
      .whenMatchedUpdateAll()
      .whenNotMatchedInsertAll()
      .execute()
    )
  else:
    df_source.write.format("delta").saveAsTable(target_table)
    

In [0]:
def get_last_processed_bronze_ingested_at(entity_name: str):
    ctrl = spark.read.table("ecomdatabricks.silver_schema.processing_control")\
            .filter(
                (f.col("layer")=="silver") &
                (f.col("entity_name")==entity_name) &
                (f.col("run_status") == "success")
            )\
            .orderBy(f.col("updated_at").desc())\
            .limit(1)

    rows = ctrl.collect()
    
    if not rows:
        return None
    
    return rows[0]["last_processed_bronze_ingested_at"]

In [0]:
def upsert_silver_control(entity_name, last_processed_bronze_run_id, last_processed_bronze_ingested_at, rows_merged):
    ctrl_df = spark.createDataFrame(
        [
            (
                "silver",
                entity_name,
                last_processed_bronze_run_id,
                last_processed_bronze_ingested_at,
                int(rows_merged),
                "SUCCESS",
                silver_run_id,
                datetime.utcnow()
            )
        ],
        schema="""
            layer string,
            entity_name string,
            last_processed_bronze_run_id string,
            last_processed_bronze_ingested_at timestamp,
            rows_merged bigint,
            run_status string,
            silver_run_id string,
            updated_at timestamp
        """
    )

    dt = DeltaTable.forName(spark, "ecomdatabricks.silver_schema.processing_control")
    (
        dt.alias("target").merge(ctrl_df.alias("s"), "target.layer=s.layer  and target.entity_name = s.entity_name")
        .whenMatchedUpdate(set={
            "last_processed_bronze_run_id": "s.last_processed_bronze_run_id",
            "last_processed_bronze_ingested_at" : "s.last_processed_bronze_ingested_at",
            "rows_merged" : "s.rows_merged",
            "run_status" : "s.run_status",
            "silver_run_id" : "s.silver_run_id",
            "updated_at" : "s.updated_at"
        })
        .whenNotMatchedInsertAll()
        .execute()
    )

In [0]:
def get_incremental_bronze(bronze_table, entity_name):
    last_ingested_at = get_last_processed_bronze_ingested_at(entity_name)
    bronze_df = spark.read.table(bronze_table)


    if last_ingested_at is None:
        return bronze_df, last_ingested_at
    
    return bronze_df.filter(f.col("bronze_ingested_at") > f.lit(last_ingested_at)), last_ingested_at

## Step 4 --Orders Incremental Processing
This cell processes **orders** from Bronze to Silver.

It does the following:
- reads only the new Bronze order rows
- cleans values like `order_status` and `order_amount`.
- keeps only the latest version per `order_id`.
- validates business rules.
- sends bad rows to quarantine.
- merges good rows into `orders_transformed`

In [0]:
df_raw = spark.read.table("ecomdatabricks.bronze_schema.orders_raw")
display(df_raw)

In [0]:
## step 4 - Orders incremental processing
## read only the Bronze order rows that silver has not processed yet.

orders_inc, last_orders_ingested_at = get_incremental_bronze(
    "ecomdatabricks.bronze_schema.orders_raw",
    "orders"
)


## counts the incremental order rows entering in this run.
orders_inc_count = orders_inc.count()
print(f"orders rows to process in the silver = {orders_inc_count}")


# Only run silver order cleaning and validation when there are new Bronze order rows.

if orders_inc_count > 0:
    # create a window that keeps the latest order record for each order_id
    order_window = Window.partitionBy("order_id").orderBy(
        f.col("updated_at").cast("timestamp").desc(),
        f.col("bronze_ingested_at").desc()
    )

    ## start the silver order cleaning pipeline. This block standardizes and deduplicates raw order records.

    orders_cleaned = (
        orders_inc
        ## Standardize order_status to upper case so values such as shipped and SHIPPED become consistent 
        .withColumn("order_status", f.upper(f.trim(f.col("order_status"))))
        .withColumn("order_status", f.when(f.col("order_status")=="", f.lit(None)).otherwise(f.col("order_status")))
        # remving formatting characters from order_amount so it can be cast to be a numeric. type
        .withColumn("order_amount", f.regexp_replace(f.col("order_amount"), r"[$, ]", ""))
        .withColumn("order_amount", f.when(f.trim(f.col("order_amount")).isin("N/A", "NULL", "??", ""), None).otherwise(f.col("order_amount")))
        .withColumn("order_amount", f.col("order_amount").cast("double"))
        .withColumn("created_at", f.to_timestamp("created_at"))
        .withColumn("updated_at", f.to_timestamp("updated_at"))
        ## Assign a row number inside each business key so we can keep only the latest version of this records
        .withColumn("row_rank", f.row_number().over(order_window))
        # keep only the latest records for each business key
        .filter(f.col("row_rank")==1).drop("row_rank")
        .withColumn("silver_run_id", f.lit(silver_run_id))
    )


    ## Merge the cleaned or validated silver dataset into it's Delta target table
    upsert_to_silver(
        orders_cleaned,
        "ecomdatabricks.silver_schema.orders_cleaned",
        "order_id"
    )

    ## Apply Silver data-quality rules to the cleaned orders records
    orders_validated = (
        orders_cleaned
        .withColumn(
            "to_be_verified_by_orders_team",
            f.when(f.col("customer_id").isNull(), "verify_customer_id")
            .when(f.col("product_id").isNull(), "verify_product_id")
            .when(f.col("order_status").isNull() | (f.trim(f.col("order_status")) ==""), "verify_order_status")
            .when(f.col("order_amount").isNull() | (f.col("order_amount") <= 0), "verify_order_amount")
            .otherwise("No Issue")
        )
        .withColumn(
            "check_order_amount",
            f.when(f.col("order_amount").isNull() | (f.col("order_amount") <= 0), f.lit(True))
            .otherwise(f.lit(False))
        )
        .withColumn("order_date", f.to_date("created_at"))
        .withColumn("order_year", f.year("created_at"))
        .withColumn("order_month", f.month("created_at"))
        .withColumn("order_day", f.dayofmonth("created_at"))
        .withColumn("order_dow", f.date_format("created_at", "E"))
    )

    ## keep only valid orders rows for the transformed silver table
    orders_good = orders_validated.filter(f.col("to_be_verified_by_orders_team") == "No Issue")

    # send invalid order rows to quarantine for manual review
    orders_bad = (
        orders_validated
        .filter(f.col("to_be_verified_by_orders_team") != "No Issue")
        .withColumn("quarantine_ts", f.current_timestamp())
    )

    ## Merge the cleaned or validated Silver dataset into it's Delta target table

    upsert_to_silver(
        orders_good,
        "ecomdatabricks.silver_schema.orders_transformed",
        "order_id"
    )


    # Append bad order rows to  the quarantine tables instea of lossing them
    orders_bad.write.format("delta").mode("append").saveAsTable("ecomdatabricks.silver_schema.orders_quarantine")



    mx_ingested = orders_inc.agg(f.max("bronze_ingested_at").alias("mx")).collect()[0]['mx']
    mx_run = (
        orders_inc.filter(f.col("bronze_ingested_at") == f.lit(mx_ingested))
        .agg(f.max("bronze_run_id").alias("mx")).collect()[0]['mx']
    )

    upsert_silver_control("orders", mx_run, mx_ingested, orders_good.count())

else:
    print("No new orders Bronze rows for Silver")

    upsert_silver_control(
        "orders",
        None,
        last_orders_ingested_at,
        orders_inc_count
    )


In [0]:
df = spark.sql("select * from ecomdatabricks.silver_schema.orders_transformed")
display(df)

## Step 5 -- Products Incremental Processing

This cell processes **Products** from Bronze to Silver

It handles:
- product name cleanup
- category standardization
- price cleanup and numeric conversion
- latest record selection per `product_id`
- data quality validation
- quarantine for bad rows
- merge into Silver current-state tables

In [0]:
%sql
select * from ecomdatabricks.bronze_schema.products_raw LIMIT 100

In [0]:
## Step 5 - products Incremental processing
# Read only the Bronze products rows that silver has not processed yet

products_inc, last_products_ingested_at = get_incremental_bronze("ecomdatabricks.bronze_schema.products_raw", "products")


# count the incremental products rows entering Silver in this run
products_inc_count = products_inc.count()
print(f"products rows_to_process_in_silver: {products_inc_count}")

if products_inc.count() >0:

    # create a window that keeps the latest products records for each product_id

    product_window = Window.partitionBy("product_id").orderBy(
        f.col("updated_at").cast("timestamp").desc(),
        f.col("bronze_ingested_at").desc()
    )

    # start the silver products-cleaning pipeline. This block standardize and deduplicates raw product records
    products_cleaned = (
        products_inc
        # Standardize product name by trimming spaces and converting text into uppercase
        .withColumn("product_name", f.upper(f.trim(f.col("product_name"))))
        .withColumn("product_name", f.regexp_replace(f.col("product_name"), r"[-_]", " "))
        .withColumn("product_name", f.when(f.col("product_name")=="", f.lit(None)).otherwise(f.col("product_name")))
        .withColumn(
            "category",
            f.when(f.upper(f.trim(f.col("category"))).contains("ELECTRNICS"), "ELECTRONICS")
            # .when(f.upper(f.trim(f.col("category")))=="FITNESS", "FITNESS")
            # .when(f.upper(f.trim(f.col("category")))=="LIFESTYLE", "LIFESTYLE")
            .otherwise(f.upper(f.trim(f.col("category"))))
        )


        ## start cleaning the product price column before converting into a numeric type
        .withColumn("price", f.trim(f.col("price")))
        .withColumn("price", f.regexp_replace(f.col("price"), r"\$", ""))
        .withColumn("price", f.regexp_replace(f.col("price"), r",", "."))
        .withColumn("price", f.regexp_replace(f.col("price"), r"\s+", ""))
        .withColumn("price", f.expr("try_cast(price as double)"))
        .withColumn("updated_at", f.to_timestamp("updated_at"))

        # assign a row number inside each business key so we can keep only the latest version of the records
        .withColumn("row_rank", f.row_number().over(product_window))
        ## keep only the latest records for each business key
        .filter(f.col("row_rank")==1)
        .drop("row_rank")
        .withColumn("silver_run_id", f.lit(silver_run_id))
    )

    ## merge the cleaned or validated silver dataset into it's Delta Target Table
    upsert_to_silver(products_cleaned, "ecomdatabricks.silver_schema.products_cleaned", "product_id")


    ##Apply Silver data-quality rules to the cleaned products records
    products_validated = (
        products_cleaned
        .withColumn(
            "to_be_verified_by_products_team",
        f.when(f.col("product_name").isNull(), "verify_product_name")
        .when(f.col("category").isNull(), "verify_category")
        .when(f.col("price").isNull() | (f.col("price") <=0 ), "verify_price")
        .otherwise("No Issue")
        )
        .withColumn(
            "check_product_price",
            f.when(f.col("price").isNull() | (f.col("price")<= 0), "invalid_price").otherwise("valid_price")
        )
    )

    # Keep only valid product rows for the transformed silver table.

    products_good = products_validated.filter(
        (f.col("to_be_verified_by_products_team") == "No Issue") &
        (f.col("check_product_price") == "valid_price")
    )

    if "price_raw" in products_good.columns:
        ## keep only valid product row for the transformed silver table
        products_good = products_good.drop("price_raw")

    
    ## send invalid product rows to the quarantine dataset for manual review

    products_bad = products_validated.filter(
        (f.col("to_be_verified_by_products_team") != "No Issue") |
        (f.col("check_product_price") == "invalid_price")
        
    ).withColumn("quarantine_ts", f.current_timestamp())

    
    ## merge the cleaned or validated silver dataset into it's Delta Target table
    upsert_to_silver(products_good, "ecomdatabricks.silver_schema.products_transformed", "product_id")

    # append the bad product rows to the quarantine table instead of lossing them
    products_bad.write.format("delta").mode("append").saveAsTable("ecomdatabricks.silver_schema.products_quarantine")


    mx_ingested = products_inc.agg(f.max("bronze_ingested_at").alias("mx")).collect()[0]["mx"]
    mx_run = products_inc.filter(f.col("bronze_ingested_at")==f.lit(mx_ingested)).agg(f.max("bronze_run_id").alias("mx")).collect()[0]["mx"]

    upsert_silver_control("products", mx_run, mx_ingested, products_good.count())

else:
    print("No new products Bronze rows for silver")
    upsert_silver_control(
        "products",
        None,
        last_products_ingested_at,
        products_inc_count
    )



In [0]:
%sql
select * from ecomdatabricks.silver_schema.products_transformed

## Step 6 -- Payment Incremental Processing

this cell processes **payments** from Bronze to Silver.

It cleans:
- `payment_status`
- `paid_amount`
- `processed_at`

Then it validates records, quarantines bad rows, and merges valid rows into the Silver transformed payments table.

In [0]:
## Step 6 - Payments Incremental processing 
# Read only the bronze payment rows that Silver has not processed yet

payments_inc, last_payment_ingested_at = get_incremental_bronze("ecomdatabricks.bronze_schema.payments_raw", "payments")
print("Payments last processed Bronze ingested_at: ", last_payment_ingested_at)


## count the incremental paymet rows entering silver in this run
payments_inc_count = payments_inc.count()
print(f"payments rows to process in silver: {payments_inc_count}")

if payments_inc_count > 0:
  ## create a window that keeps the latest payments records for each payment_id
  payment_window = Window.partitionBy("payment_id").orderBy(
    f.col("processed_at").cast("timestamp").desc(),
    f.col("bronze_ingested_at").desc()
  )


  ## start the silver payment-cleaning pipeline. This bloc standardize and deduplicates raw payment records.

  payment_cleaned = (
    payments_inc
    .withColumn("payment_status", f.upper(f.trim(f.col("payment_status"))))
    .withColumn("payment_status", f.when(f.col("payment_status")=="",f.lit(None)).otherwise(f.col("payment_status")))

    ## start cleaning the payment amount field before converting it into numeric.

    .withColumn("paid_amount", f.trim(f.col("paid_amount")))
    .withColumn("paid_amount", f.regexp_replace(f.col("paid_amount"), r"\$", ""))
    .withColumn("paid_amount", f.regexp_replace(f.col("paid_amount"), r",", "."))
    .withColumn("paid_amount", f.regexp_replace(f.col("paid_amount"), r"\s+", ""))
    .withColumn("paid_amount", f.expr("try_cast(paid_amount as double)"))
    .withColumn("processed_at", f.to_timestamp("processed_at"))

    # Assign a row number inside each business key so we can keep only the latest version of that record.
    .withColumn("row_rank", f.row_number().over(payment_window))
    # keep only the latest record for each business key
    .filter(f.col("row_rank")==1)
    .drop("row_rank")
    .withColumn("silver_run_id", f.lit(silver_run_id))
  )

  ## Merge the cleaned or validated Silver dataset into it's Delta Target table
  upsert_to_silver(payment_cleaned, "ecomdatabricks.silver_schema.payments_cleaned", "payment_id")

  ## apply silver data-quality rules to the cleaned payment records
  payment_validated = (
    payment_cleaned
    .withColumn(
      "to_be_verified_by_payment_team",
      f.when(f.col("order_id").isNull(), "verify_order_id")
      .when(f.col("payment_status").isNull(), "verify_payment_status")
      .when(f.col("paid_amount").isNull() | (f.col("paid_amount")<= 0), "verify_paid_amount")
      .otherwise("No Issue")       
    ) 
    .withColumn("check_paid_amount",
                f.when(f.col("paid_amount").isNull() | (f.col("paid_amount") <= 0), f.lit(True)).otherwise(f.lit(False))
      )
  )

  ## Keep only valid payment rows for the transformed silver tables

  payments_good = payment_validated.filter(f.col("to_be_verified_by_payment_team")=="No Issue")

  # send invalid payment rows to quarantine table instead of lossing them
  payments_bad = payment_validated.filter(f.col("to_be_verified_by_payment_team")!="No Issue").withColumn("quarantine_ts", f.current_timestamp())

  ## merge the cleaned or validated Silver dataset into it's Delta target tables
  upsert_to_silver(payments_good, "ecomdatabricks.silver_schema.payments_transformed", "payment_id")

  ## Append bad payment rows to the quarantine table instead of losing
  payments_bad.write.format("delta").mode("append").saveAsTable("ecomdatabricks.silver_schema.payments_quarantine")

  mx_ingested = payments_inc.agg(f.max("bronze_ingested_at").alias("mx")).collect()[0]["mx"]

  mx_run = payments_inc.filter(f.col("bronze_ingested_at")==f.lit(mx_ingested)).agg(f.max("bronze_run_id").alias("mx")).collect()[0]["mx"]

  upsert_silver_control("payments", mx_run, mx_ingested, payments_inc_count)

else:
  print("No new payments Bronze rows for silver")
  upsert_silver_control(
      "payments",
      None,
      last_payment_ingested_at,
      payments_inc_count
  )


In [0]:
%sql
select * from ecomdatabricks.silver_schema.payments_quarantine

## Step 7 -- Quick Validation


In [0]:
print("products transformed count: ", spark.sql("select count(*) from ecomdatabricks.silver_schema.products_transformed").collect()[0][0])

print("Orders transformed count: ", spark.sql("select count(*) from ecomdatabricks.silver_schema.orders_transformed").collect()[0][0])

print("Payments transformed count: ", spark.sql("select count(*) from ecomdatabricks.silver_schema.payments_transformed").collect()[0][0])



display(spark.table("ecomdatabricks.silver_schema.processing_control").orderBy("entity_name"))